# Notebook 8: Physical Realizability (EOT)

Evaluate physical realizability of attacks using Expectation Over Transformations.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from data_loader import SensorDataLoader
from attacks.camera_attacks import CameraAdversarialAttacker, AttackType
from attacks.physical_eot import PhysicalRealizabilityModel

%matplotlib inline

## 8.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
DATA_PATH = f'../data/sensor_fusion_dataset/{SCENARIO}'

loader = SensorDataLoader(DATA_PATH)
detections = loader.load_all_detections()
ir_detections = detections[3]

print(f"IR Camera: {len(ir_detections)} detections")

## 8.2 Initialize Physical Model

In [ ]:
# Different maritime conditions
conditions = {
    'Calm': {'wave_height': 0.5, 'rain_rate': 0.0, 'fog_visibility': 10000, 'sun_glint_angle': 90},
    'Moderate Waves': {'wave_height': 2.0, 'rain_rate': 0.0, 'fog_visibility': 10000, 'sun_glint_angle': 90},
    'Heavy Rain': {'wave_height': 0.5, 'rain_rate': 10.0, 'fog_visibility': 10000, 'sun_glint_angle': 90},
    'Fog': {'wave_height': 0.5, 'rain_rate': 0.0, 'fog_visibility': 500, 'sun_glint_angle': 90},
    'Sun Glint': {'wave_height': 0.5, 'rain_rate': 0.0, 'fog_visibility': 10000, 'sun_glint_angle': 10},
}

models = {name: PhysicalRealizabilityModel(**params) for name, params in conditions.items()}
print(f"Initialized {len(models)} environment conditions")

## 8.3 Generate Attack

In [ ]:
attacker = CameraAdversarialAttacker(epsilon=0.05)
attacked = attacker.attack_detections(ir_detections.copy(), AttackType.FGSM, sensor_id=3)

# Compute perturbation
perturbation = attacked['bearing'].values - ir_detections['bearing'].values
print(f"Perturbation range: [{perturbation.min():.4f}, {perturbation.max():.4f}] rad")
print(f"Mean perturbation: {np.mean(np.abs(perturbation)):.4f} rad")

## 8.4 Evaluate Realizability Scores

In [ ]:
# Evaluate each condition
realizability_results = {}

for condition_name, model in models.items():
    scores = []
    for i in range(min(100, len(ir_detections))):
        det = {
            'x_piren': ir_detections.iloc[i]['x_piren'],
            'y_piren': ir_detections.iloc[i]['y_piren'],
            'time': ir_detections.iloc[i]['time']
        }
        score = model.evaluate_realizability(det, perturbation[i])
        scores.append(score)
    
    realizability_results[condition_name] = {
        'mean': np.mean(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores)
    }
    print(f"{condition_name:20s}: mean={np.mean(scores):.3f}, std={np.std(scores):.3f}")

## 8.5 Visualize Realizability by Condition

In [ ]:
conditions_names = list(realizability_results.keys())
means = [realizability_results[c]['mean'] for c in conditions_names]
stds = [realizability_results[c]['std'] for c in conditions_names]

plt.figure(figsize=(12, 6))
bars = plt.bar(conditions_names, means, yerr=stds, capsize=5, 
               color=['green', 'yellow', 'orange', 'gray', 'red'], alpha=0.7)
plt.axhline(y=0.5, color='k', linestyle='--', label='Threshold (0.5)')
plt.ylabel('Realizability Score')
plt.title('Physical Realizability of FGSM Attack Under Different Conditions')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8.6 EOT with Multiple Samples

In [ ]:
# Test EOT with varying number of transformation samples
sample_counts = [1, 5, 10, 20, 50]
eot_results = []

model = PhysicalRealizabilityModel(wave_height=1.0, rain_rate=5.0, fog_visibility=2000)

for n_samples in sample_counts:
    scores = []
    for i in range(min(50, len(ir_detections))):
        det = {
            'x_piren': ir_detections.iloc[i]['x_piren'],
            'y_piren': ir_detections.iloc[i]['y_piren'],
            'time': ir_detections.iloc[i]['time']
        }
        # Simulate EOT by averaging multiple evaluations
        sample_scores = [model.evaluate_realizability(det, perturbation[i]) for _ in range(n_samples)]
        scores.append(np.mean(sample_scores))
    
    eot_results.append({'n_samples': n_samples, 'mean_score': np.mean(scores)})
    print(f"n_samples={n_samples:3d}: mean realizability={np.mean(scores):.3f}")

# Plot convergence
plt.figure(figsize=(10, 6))
plt.plot([r['n_samples'] for r in eot_results], [r['mean_score'] for r in eot_results], 
         'bo-', linewidth=2, markersize=10)
plt.xlabel('Number of EOT Samples')
plt.ylabel('Mean Realizability Score')
plt.title('EOT Convergence: Realizability vs Sample Count')
plt.grid(True)
plt.show()

## 8.7 Attack Success vs Realizability Trade-off

In [ ]:
# Test different epsilon values
epsilons = np.linspace(0.01, 0.2, 10)
model = PhysicalRealizabilityModel(wave_height=1.0, rain_rate=2.0)

trade_off = []

for eps in epsilons:
    att = CameraAdversarialAttacker(epsilon=eps)
    attacked = att.attack_detections(ir_detections.copy(), AttackType.FGSM, sensor_id=3)
    pert = attacked['bearing'].values - ir_detections['bearing'].values
    
    # Attack success (mean perturbation)
    attack_success = np.mean(np.abs(pert))
    
    # Realizability
    scores = []
    for i in range(min(50, len(ir_detections))):
        det = {
            'x_piren': ir_detections.iloc[i]['x_piren'],
            'y_piren': ir_detections.iloc[i]['y_piren'],
            'time': ir_detections.iloc[i]['time']
        }
        scores.append(model.evaluate_realizability(det, pert[i]))
    
    realizability = np.mean(scores)
    trade_off.append({'epsilon': eps, 'attack_success': attack_success, 'realizability': realizability})

# Plot trade-off
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([t['epsilon'] for t in trade_off], [t['attack_success'] for t in trade_off], 
         'ro-', linewidth=2, markersize=8)
ax1.set_xlabel('Epsilon')
ax1.set_ylabel('Attack Success (rad)')
ax1.set_title('Attack Success vs Epsilon')
ax1.grid(True)

ax2.plot([t['epsilon'] for t in trade_off], [t['realizability'] for t in trade_off], 
         'bo-', linewidth=2, markersize=8)
ax2.set_xlabel('Epsilon')
ax2.set_ylabel('Realizability Score')
ax2.set_title('Realizability vs Epsilon')
ax2.grid(True)

plt.suptitle('Attack Success vs Physical Realizability Trade-off', fontsize=14)
plt.tight_layout()
plt.show()